In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/raw/raw_data.csv")
def split_train_test_bags(df_raw, test_bag_count=6, seed=42):
    """
    Splits raw telemetry DataFrame into Train and Test sets by entire .bag files.
    Ensures both sets contain representative fault samples without data leakage.
    """
    # 1. Inspect total rows and fault presence per bag file
    bag_summary = df_raw.groupby('bag').agg(
        total_samples=('timestamp_ns', 'count'),
        has_engine_fault=('engine_fault', lambda x: (x > 0).any()),
        has_aileron_fault=('aileron_fault', lambda x: (x > 0).any()),
        has_elevator_fault=('elevator_fault', lambda x: (x > 0).any()),
        has_rudder_fault=('rudder_fault', lambda x: (x > 0).any())
    ).reset_index()
    
    # Flag bags that contain ANY fault state
    bag_summary['is_faulty'] = (
        bag_summary['has_engine_fault'] | 
        bag_summary['has_aileron_fault'] | 
        bag_summary['has_elevator_fault'] | 
        bag_summary['has_rudder_fault']
    )
    
    # 2. Separate healthy and faulty bags
    faulty_bags = bag_summary[bag_summary['is_faulty']]['bag'].values
    healthy_bags = bag_summary[~bag_summary['is_faulty']]['bag'].values
    
    print(f"Total Bags Found: {len(bag_summary)}")
    print(f"Healthy Bags: {len(healthy_bags)} | Faulty Bags: {len(faulty_bags)}")
    
    # 3. Stratified selection (e.g., 4 faulty bags + 2 healthy bags for a strong test set)
    np.random.seed(seed)
    selected_faulty_test = np.random.choice(faulty_bags, size=min(4, len(faulty_bags)), replace=False)
    selected_healthy_test = np.random.choice(healthy_bags, size=min(2, len(healthy_bags)), replace=False)
    
    test_bags = list(selected_faulty_test) + list(selected_healthy_test)
    
    # 4. Filter DataFrame into raw Train and Test splits
    df_raw_test = df_raw[df_raw['bag'].isin(test_bags)].copy().reset_index(drop=True)
    df_raw_train = df_raw[~df_raw['bag'].isin(test_bags)].copy().reset_index(drop=True)
    
    print("\n--- SPLIT COMPLETE ---")
    print(f"Selected Test Bags (6 total): {test_bags}")
    print(f"Raw Train Shape: {df_raw_train.shape} ({len(df_raw_train)/len(df_raw):.1%})")
    print(f"Raw Test Shape:  {df_raw_test.shape} ({len(df_raw_test)/len(df_raw):.1%})")
    
    return df_raw_train, df_raw_test, test_bags

# --- Execute Split ---
df_raw_train, df_raw_test, test_bag_list = split_train_test_bags(df_raw)

Total Bags Found: 36
Healthy Bags: 7 | Faulty Bags: 29

--- SPLIT COMPLETE ---
Selected Test Bags (6 total): ['carbonZ_2018-10-18-11-04-35.bag', 'carbonZ_2018-09-11-15-06-34.bag', 'carbonZ_2018-09-11-14-41-38.bag', 'carbonZ_2018-10-05-15-55-10.bag', 'carbonZ_2018-07-18-12-10-11.bag', 'carbonZ_2018-09-11-14-42-06.bag']
Raw Train Shape: (132937, 24) (73.5%)
Raw Test Shape:  (47875, 24) (26.5%)


In [2]:
print("\nSaving files to disk...")
df_raw_train.to_csv('train_raw.csv', index=False)
df_raw_test.to_csv('test_raw.csv', index=False)
print("Saved 'train_raw.csv' and 'test_raw.csv' successfully!")


Saving files to disk...
Saved 'train_raw.csv' and 'test_raw.csv' successfully!
